# 03 · Traduction de référence avec NLLB-200 (français ↔ bambara)

**But.** Mesurer ce que vaut **NLLB-200 tel quel**, sans aucun entraînement de notre part. Ce score est la **référence à battre** avec le fine-tuning (notebook 05). Sans référence, on ne saurait pas si l'entraînement a servi à quelque chose.

**Le modèle.** `facebook/nllb-200-distilled-600M` : un modèle de traduction de Meta qui couvre 200 langues, dont le bambara (code `bam_Latn`) et le français (`fra_Latn`). « 600M » = 600 millions de paramètres.

| | |
|---|---|
| **Prérequis** | avoir fait le notebook 01 (il faut `data/processed/test.tsv`) |
| **Matériel** | **GPU NVIDIA fortement recommandé**. Sur processeur, ça marche mais très lentement (le notebook réduit alors le nombre de phrases) |
| **Internet** | oui au premier lancement : le modèle pèse environ 2,5 Go et est gardé en cache ensuite |
| **Ce que tu obtiens** | des scores BLEU / chrF et des traductions dans `results/` |

### Comment on mesure la qualité

- **BLEU** compte les mots (et suites de mots) communs entre la traduction du modèle et la traduction humaine de référence. Échelle de 0 à 100. Pour une langue peu dotée en données, quelques points de BLEU sont déjà normaux.
- **chrF** fait la même chose avec des suites de *caractères*. Il est plus indulgent et plus fiable pour des langues riches en formes de mots comme le bambara.
- **Le plancher** : on mesure aussi un « système » qui recopie simplement la phrase d'entrée. Il donne un petit score (noms propres, chiffres). Un vrai système doit faire nettement mieux.
- **Limite** : ces scores punissent toute formulation différente de la référence, même correcte. Ils *sous-estiment* donc un peu la qualité réelle.

## 1. Se repérer dans les dossiers

**Objectif.** Retrouver les dossiers du projet.

**Ce que fait la cellule.** Détecte la racine du projet et crée les dossiers manquants.

**Ce que tu dois voir.** Une ligne `Racine du projet : ...`.

In [1]:
import os
from pathlib import Path

# Ces deux réglages évitent des avertissements inutiles de Hugging Face.
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Racine du projet = le dossier qui contient "data", "models", "results", "src"...
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if not (ROOT / "src").exists():
    raise RuntimeError(
        f"Dossier du projet introuvable depuis {cwd}. "
        "Lance Jupyter avec bash lancer_jupyter.sh (ou depuis le dossier du projet)."
    )

DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
MODELS = ROOT / "models"
RESULTS = ROOT / "results"
for folder in (DATA_RAW, DATA_PROC, MODELS, RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

print("Racine du projet :", ROOT)

Racine du projet : /mnt/c/Users/RobotsMali-Victus/Desktop/nlp-bambara/nlp-bambara-wsl


## 2. Charger le jeu de test

**Objectif.** Récupérer les phrases sur lesquelles on va évaluer (jamais vues à l'entraînement).

**Ce que fait la cellule.** Lit `data/processed/test.tsv`, produit par le notebook 01.

**Ce que tu dois voir.** `1499 paires de test` (environ) et les trois premières lignes.

> **Attention.** Si le fichier est introuvable, lance d'abord le notebook 01.

In [2]:
import pandas as pd

TEST_PATH = DATA_PROC / "test.tsv"
if not TEST_PATH.exists():
    raise FileNotFoundError(f"{TEST_PATH} introuvable : lance d'abord le notebook 01.")

test = pd.read_csv(TEST_PATH, sep="\t", encoding="utf-8")
print(len(test), "paires de test")
test.head(3)

1499 paires de test


,fr,bam
0,Affaire à suivre Drissa Togola Le Premier mini...,Ko min tɔ bɛ kɔ - Dirisa Togola - Minisiriɲɛmɔ...
1,L’objectif de cette visite de courtoisie à la ...,Nin balimayataama ka taa bɔ Kunnafonidalaw ka ...
2,Le premier ministre a profité l’occasion pour ...,Minisiriɲɛmɔgɔ ye nin ko kɛ sababu ye ka Jaman...


## 3. Détecter le matériel

**Objectif.** Savoir si le calcul se fera sur la carte graphique (rapide) ou sur le processeur (lent).

**Ce que fait la cellule.** Interroge PyTorch. Règle aussi la taille des lots de traduction selon la mémoire vidéo disponible : plus la carte a de mémoire, plus on traduit de phrases à la fois.

**Ce que tu dois voir.** Soit `GPU détecté : NVIDIA ... (x Go de mémoire vidéo)`, soit un message disant qu'aucun GPU n'est utilisable.

> **Attention.** Si tu as une carte NVIDIA mais que le notebook dit « Aucun GPU », PyTorch a été installé en version processeur. Relance `bash installer.sh` et choisis l'option NVIDIA.

In [3]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU détecté : {gpu_name} ({vram_gb:.1f} Go de mémoire vidéo)")
else:
    device = "cpu"
    vram_gb = 0.0
    print("Aucun GPU utilisable : le calcul se fera sur le processeur (CPU), beaucoup plus lentement.")
    print("Si tu as une carte NVIDIA, relance bash installer.sh et choisis l'option NVIDIA.")
print("PyTorch :", torch.__version__, "| CUDA compilée :", torch.version.cuda)

# Taille des lots pour traduire : prudente selon la mémoire vidéo disponible.
if device == "cuda":
    TRANSLATE_BATCH = 32 if vram_gb >= 8 else 16 if vram_gb >= 4 else 8
else:
    TRANSLATE_BATCH = 8

GPU détecté : NVIDIA GeForce RTX 2050 (4.0 Go de mémoire vidéo)
PyTorch : 2.14.1+cu126 | CUDA compilée : 12.6


## 4. Charger NLLB

**Objectif.** Mettre en mémoire le modèle de traduction et son « tokenizer » (le programme qui découpe le texte en petits morceaux que le modèle comprend).

**Ce que fait la cellule.** Télécharge le modèle depuis Hugging Face au premier lancement (environ 2,5 Go, plusieurs minutes selon ta connexion), puis le charge. Sur GPU, il est converti en demi-précision (`half`) : deux fois moins de mémoire, plus rapide, sans perte notable pour traduire.

**Ce que tu dois voir.** Deux lignes qui affichent les identifiants numériques des codes de langue `fra_Latn` et `bam_Latn` (des nombres, pas `None` ni une erreur).

> **Attention.** La première exécution est longue. Les suivantes sont rapides : le modèle est gardé dans le cache de Hugging Face (dossier `~/.cache/huggingface`).

In [4]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()  # coupe un avertissement répétitif sans effet sur les résultats

MODEL_NAME = "facebook/nllb-200-distilled-600M"
FR, BAM = "fra_Latn", "bam_Latn"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
if device == "cuda":
    model = model.half()      # fp16 d'abord sur le processeur : évite un pic de mémoire vidéo
model = model.to(device)
model.eval()

for code_ in (FR, BAM):
    print(code_, "->", tokenizer.convert_tokens_to_ids(code_))

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

fra_Latn -> 256057
bam_Latn -> 256022


## 5. La fonction de traduction, et un essai

**Objectif.** Écrire une fonction qui traduit une liste de phrases, puis vérifier qu'elle marche sur trois exemples.

**Ce que fait la cellule.** Les phrases sont triées par longueur (traduire ensemble des phrases de longueur voisine est plus rapide), traduites par lots, puis remises dans l'ordre d'origine. On utilise la *recherche par faisceaux* (`num_beams=4`) : le modèle explore 4 traductions possibles à la fois et garde la meilleure. `forced_bos_token_id` impose la langue de sortie.

**Ce que tu dois voir.** Trois traductions en bambara. Elles ne seront pas parfaites : c'est justement ce qu'on mesure.

> **Attention.** Ces réglages (4 faisceaux, 256 jetons maximum) sont les **mêmes** dans les notebooks 04 et 05, pour que les comparaisons soient honnêtes. Ne les change pas entre deux expériences que tu veux comparer.

In [5]:
import torch

def translate(texts, src, tgt, batch_size=None, num_beams=4, max_new_tokens=256):
    batch_size = batch_size or TRANSLATE_BATCH
    tokenizer.src_lang = src
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    outputs = [None] * len(texts)
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        batch = [texts[i] for i in idx]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
        with torch.no_grad():
            generated = model.generate(
                **enc,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt),
                num_beams=num_beams,
                max_new_tokens=max_new_tokens,
            )
        for i, text in zip(idx, tokenizer.batch_decode(generated, skip_special_tokens=True)):
            outputs[i] = text
    return outputs

phrases = ["Bonjour, comment allez-vous ?", "Le marché ouvre à huit heures.", "Merci beaucoup pour votre aide."]
for p, t in zip(phrases, translate(phrases, FR, BAM)):
    print(p, "->", t)

Bonjour, comment allez-vous ? -> Bonjour, i bɛ cogo di?
Le marché ouvre à huit heures. -> Sɔrɔda bɛ yɛlɛ lɛrɛ segin na.
Merci beaucoup pour votre aide. -> Ne bɛ aw fo kosɛbɛ aw ka dɛmɛni kosɔn.


## 6. Traduire le jeu de test (dans les deux sens)

**Objectif.** Obtenir les traductions du modèle pour un échantillon du test, en français → bambara puis en bambara → français.

**Ce que fait la cellule.** `LIMIT` fixe le nombre de phrases évaluées : **300 sur GPU** (quelques minutes), seulement **40 sur processeur** (sinon ce serait interminable). Mets `LIMIT = None` pour évaluer tout le test (~1 500 phrases) : à faire une seule fois, pour le score final. La cellule affiche le temps pris dans chaque sens.

**Ce que tu dois voir.** Deux durées, par exemple `fr -> bam : 94 s` et `bam -> fr : 35 s` pour 300 phrases lors de ton essai sur Colab. Le sens français → bambara est plus long car les phrases bambara générées sont plus longues en jetons.

> **Attention.** Sur processeur, même 40 phrases peuvent prendre plusieurs minutes. Ne ferme pas le notebook : regarde le temps qui s'écoule.

In [6]:
import time

LIMIT = 300 if device == "cuda" else 40
subset = test.head(LIMIT) if LIMIT else test
print(len(subset), "phrases")

t0 = time.time()
hyp_fr_bam = translate(subset["fr"].tolist(), FR, BAM)
print(f"fr -> bam : {time.time() - t0:.0f} s")

t0 = time.time()
hyp_bam_fr = translate(subset["bam"].tolist(), BAM, FR)
print(f"bam -> fr : {time.time() - t0:.0f} s")

300 phrases
fr -> bam : 125 s
bam -> fr : 60 s


## 7. Calculer les scores

**Objectif.** Chiffrer la qualité du modèle de base, avec le plancher pour comparaison.

**Ce que fait la cellule.** Utilise la bibliothèque `sacrebleu` (outil standard de la recherche) pour calculer BLEU et chrF dans les deux sens. Ajoute la ligne « copie de la source » : le score qu'on obtient en recopiant la phrase d'entrée.

**Ce que tu dois voir.** Un tableau de 4 lignes. Lors de ton premier essai sur Colab (300 phrases), les scores étaient, pour fr → bam : BLEU 5,39 et chrF 27,42 (plancher : 0,62 et 13,36) ; pour bam → fr : BLEU 9,66 et chrF 35,06. Tes chiffres peuvent différer un peu selon le nombre de phrases et le matériel.

> **Attention.** Un résultat nettement au-dessus du plancher mais encore bas est normal pour une langue peu dotée en données. Il y a risque de *contamination* : NLLB a pu voir des textes proches de MAFAND à son entraînement, donc ses scores sont peut-être un peu optimistes.

In [7]:
import sacrebleu

def score(hyps, refs):
    return {
        "bleu": round(sacrebleu.corpus_bleu(hyps, [refs]).score, 2),
        "chrf": round(sacrebleu.corpus_chrf(hyps, [refs]).score, 2),
    }

refs_bam = subset["bam"].tolist()
refs_fr = subset["fr"].tolist()

table = pd.DataFrame([
    {"direction": "fr -> bam", "système": "NLLB-200 600M", **score(hyp_fr_bam, refs_bam)},
    {"direction": "fr -> bam", "système": "copie de la source (plancher)", **score(subset["fr"].tolist(), refs_bam)},
    {"direction": "bam -> fr", "système": "NLLB-200 600M", **score(hyp_bam_fr, refs_fr)},
    {"direction": "bam -> fr", "système": "copie de la source (plancher)", **score(subset["bam"].tolist(), refs_fr)},
])
table

,direction,système,bleu,chrf
0,fr -> bam,NLLB-200 600M,5.46,27.38
1,fr -> bam,copie de la source (plancher),0.62,13.36
2,bam -> fr,NLLB-200 600M,9.67,35.10
3,bam -> fr,copie de la source (plancher),0.62,12.67


## 8. Lire les traductions toi-même

**Objectif.** Aucun score ne remplace un locuteur qui lit. Ton avis compte plus que le BLEU.

**Ce que fait la cellule.** Affiche 6 phrases tirées au hasard avec la traduction humaine de référence et celle de NLLB.

**Ce que tu dois voir.** Des traductions souvent compréhensibles mais qui disent les choses autrement, parfois avec une information perdue ou ajoutée.

> **Attention.** Pour chaque phrase, demande-toi : le **sens** est-il correct ? Une formulation différente de la référence n'est pas une erreur ; un sens perdu en est une. Note les vrais problèmes dans `results/errors_analysis.md`.

In [8]:
pd.set_option("display.max_colwidth", 140)
view = subset.assign(nllb_bam=hyp_fr_bam, nllb_fr=hyp_bam_fr)
for _, r in view.sample(min(6, len(view)), random_state=0).iterrows():
    print("FR       :", r["fr"])
    print("REF  bam :", r["bam"])
    print("NLLB bam :", r["nllb_bam"])
    print()

FR       : Ces derniers pensent que c’est juste un pilotage à vue.
REF  bam : Olu kɔni b’a miiri ko kunfɛtaa dama de don.
NLLB bam : U b'a miiri ko o ye ɲɛminɛ dɔrɔn de ye.

FR       : Ce qui crée souvent une confusion dans la circulation.
REF  bam : O bɛ wolo ɲagami na, waati dɔw la, sirabakanboli la.
NLLB bam : O bɛ mɔgɔ caman hakili ɲagami.

FR       : Sur instruction du Président de la transition, nous avons décidé de donner une impulsion, un processus d’accélération de la transition tout en cherchant l’adhésion des maliens.
REF  bam : cɛmancɛlafanga perezidan ka yamaruyali kɔnɔ, an y’a ŋaniya ka galabu don kɛ. ka cɛmancɛ fanga taabolow sennateliya, ni ka to malidenw ka sendonni ɲini na.
NLLB bam : K'an to Mali jamanakuntigi bolo, an y'a latigɛ ka dɔ fara an ka fanga kan, ka dɔ fara a kan, k'a sɔrɔ an tun b'a fɛ Mali jamanakuntigiw ka fara an kan.

FR       : Avant d’inviter les consommateurs à signaler tout contrevenant auprès des services de la Direction Générale du Commerce de l

## 9. Où le modèle est-il le plus faible ?

**Objectif.** Voir si la qualité dépend de la longueur de la phrase.

**Ce que fait la cellule.** Calcule un score chrF pour *chaque* phrase, puis en fait la moyenne par tranche de longueur de la phrase source.

**Ce que tu dois voir.** Un petit tableau (nombre de phrases et chrF moyen par tranche). Les très courtes phrases et les très longues sont souvent les plus difficiles.

> **Attention.** Avec seulement 40 phrases (sur processeur), les tranches contiennent très peu d'exemples : ne tire pas de conclusion.

In [9]:
from sacrebleu.metrics import CHRF

chrf = CHRF()
view["chrf_fr_bam"] = [chrf.sentence_score(h, [r]).score for h, r in zip(hyp_fr_bam, refs_bam)]
view["longueur"] = view["fr"].str.split().str.len()
view["tranche"] = pd.cut(view["longueur"], [0, 10, 20, 35, 1000], labels=["≤10 mots", "11-20", "21-35", ">35"])
view.groupby("tranche", observed=True)["chrf_fr_bam"].agg(["count", "mean"]).round(1)

,count,mean
tranche,,
≤10 mots,70,21.9
11-20,116,27.0
21-35,75,28.9
>35,39,32.0


## 10. Sauvegarder les résultats

**Objectif.** Garder une trace de ce baseline pour le comparer au modèle fine-tuné.

**Ce que fait la cellule.** Écrit toutes les traductions dans `results/predictions_nllb_baseline.tsv` et ajoute deux lignes (une par sens) au journal `results/metrics.csv`.

**Ce que tu dois voir.** Deux messages confirmant les chemins des fichiers.

> **Attention.** Si tu relances la cellule, elle ajoute de nouvelles lignes au journal (elle n'écrase pas les anciennes) : c'est voulu, ça garde l'historique de tes essais.

In [10]:
from datetime import datetime

view[["fr", "bam", "nllb_bam", "nllb_fr"]].to_csv(RESULTS / "predictions_nllb_baseline.tsv", sep="\t", index=False, encoding="utf-8")

metrics_path = RESULTS / "metrics.csv"
new = not metrics_path.exists() or metrics_path.stat().st_size == 0
with metrics_path.open("a", encoding="utf-8", newline="") as f:
    if new:
        f.write("date,experiment,model,direction,n_sentences,bleu,chrf\n")
    now = datetime.now().isoformat(timespec="seconds")
    for direction, hyps, refs in (("fr-bam", hyp_fr_bam, refs_bam), ("bam-fr", hyp_bam_fr, refs_fr)):
        s = score(hyps, refs)
        f.write(f"{now},baseline,{MODEL_NAME},{direction},{len(hyps)},{s['bleu']},{s['chrf']}\n")

print("Prédictions :", RESULTS / "predictions_nllb_baseline.tsv")
print("Scores ajoutés à :", metrics_path)

Prédictions : /mnt/c/Users/RobotsMali-Victus/Desktop/nlp-bambara/nlp-bambara-wsl/results/predictions_nllb_baseline.tsv
Scores ajoutés à : /mnt/c/Users/RobotsMali-Victus/Desktop/nlp-bambara/nlp-bambara-wsl/results/metrics.csv


## Suite

1. Remplis `results/errors_analysis.md` avec des exemples d'erreurs réelles (noms propres, vocabulaire, sens perdu, phrases répétées ou tronquées). Tu comprends la langue : c'est ton avantage.
2. **Notebook 05** : fine-tuning de NLLB sur `train.tsv`, puis comparaison avant / après (**GPU NVIDIA requis**, voir le guide).

*Données : MAFAND-MT, Masakhane, CC BY-NC-4.0. Modèle : NLLB-200, Meta (CC BY-NC 4.0).*